#  📊  **Ajuste y Preprocesamiento de Tipos de Datos**


---

En este notebook se realiza el proceso de ajuste y preprocesamiento de los tipos de datos del conjunto de datos utilizado en el proyecto. 

Durante el desarrollo del notebook se identifican las variables numéricas y categóricas, se realizan conversiones de tipos de datos, se corrigen inconsistencias y se preparan las columnas a su tipo de dato correspondiente.

El objetivo principal es obtener un dataset limpio, estructurado y compatible para realizar un buen EDA (Analisís Exploratorio de Datos)

## **Librerias**

In [1]:
import pyarrow as pa
from pathlib import Path
import numpy as np
import pandas as pd


Llamamos los datos con la ruta correspondiente

In [2]:

DATA_DIR = Path.cwd().parent / "data" / "processed"
print(DATA_DIR)

c:\Users\belac\OneDrive\Documentos\Universidad\Ciencia de Datos y ML\Mlops\credit-risk-classification\data\processed


In [3]:
datos_cooperativa = pd.read_parquet(DATA_DIR / "01_datos_crudos_cooperativa.parquet")

In [4]:
datos_cooperativa.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12910 entries, 0 to 12909
Data columns (total 28 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   cartera             12910 non-null  object 
 1   plazo               12910 non-null  int64  
 2   vinculacion         12910 non-null  int64  
 3   valor_cuota         12910 non-null  float64
 4   valor_prestamo      12910 non-null  float64
 5   saldo_capital       12910 non-null  float64
 6   saldo_interes       12910 non-null  int64  
 7   aportes             12910 non-null  int64  
 8   garantias           12910 non-null  int64  
 9   valorgarantia       12910 non-null  int64  
 10  reestr              12910 non-null  int64  
 11  ctasahorros         12910 non-null  float64
 12  edad                12910 non-null  float64
 13  tipoasociado        12910 non-null  int64  
 14  estado_cliente      12910 non-null  int64  
 15  departamento        12909 non-null  object 
 16  ciud

## ➜ **Conversión de Datos y Ajustes de Tipo de Dato**

### **Variables Categóricas**

Las variables categoricas se encuentran en tipo **string**, por lo que vamos a conventirlas a tipo categoricas porque los modelos no trabajan directamente con texto

In [5]:
print(datos_cooperativa.select_dtypes(include=['object', 'string']))

                    cartera departamento       ciudad
0      consumo_sin_libranza    antioquia  bolivariana
1      consumo_sin_libranza    antioquia     envigado
2      consumo_sin_libranza    antioquia     medellin
3      consumo_sin_libranza    antioquia     rionegro
4      consumo_sin_libranza    antioquia     rionegro
...                     ...          ...          ...
12905  consumo_con_libranza    antioquia       retiro
12906  consumo_con_libranza    antioquia        campo
12907  consumo_con_libranza    antioquia     medellin
12908  consumo_con_libranza    antioquia     medellin
12909  consumo_sin_libranza    antioquia     medellin

[12910 rows x 3 columns]


In [6]:
cols_categoricas = ['cartera', 'departamento', 'ciudad']
datos_cooperativa[cols_categoricas] = datos_cooperativa[cols_categoricas].astype('category')

Verificamos que los tipos de datos en el dataset se hayan transformado correctamente después de realizar la conversión correspondiente.

In [7]:
datos_cooperativa.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12910 entries, 0 to 12909
Data columns (total 28 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   cartera             12910 non-null  category
 1   plazo               12910 non-null  int64   
 2   vinculacion         12910 non-null  int64   
 3   valor_cuota         12910 non-null  float64 
 4   valor_prestamo      12910 non-null  float64 
 5   saldo_capital       12910 non-null  float64 
 6   saldo_interes       12910 non-null  int64   
 7   aportes             12910 non-null  int64   
 8   garantias           12910 non-null  int64   
 9   valorgarantia       12910 non-null  int64   
 10  reestr              12910 non-null  int64   
 11  ctasahorros         12910 non-null  float64 
 12  edad                12910 non-null  float64 
 13  tipoasociado        12910 non-null  int64   
 14  estado_cliente      12910 non-null  int64   
 15  departamento        12909 non-null  

Nuestro dataset no presenta variables categóricas ordinales, por lo tanto, en la etapa de transformación de datos mediante *pipelines* solo se prepararán las variables categóricas nominales.


In [8]:
# Visualizamos las categorias que tiene cada columna

print(datos_cooperativa["cartera"].unique())
print(datos_cooperativa["departamento"].unique())
print(datos_cooperativa["ciudad"].unique())

['consumo_sin_libranza', 'vivienda', 'consumo_con_libranza']
Categories (3, object): ['consumo_con_libranza', 'consumo_sin_libranza', 'vivienda']
['antioquia', 'valle_del_cauca', 'quindio', NaN, 'bogotá', ..., 'meta', 'norte_de_santander', 'magdalena', 'sucre', 'la_guajira']
Length: 24
Categories (23, object): ['antioquia', 'arauca', 'atlántico', 'bogotá', ..., 'santander', 'sucre', 'tolima', 'valle_del_cauca']
['bolivariana', 'envigado', 'medellin', 'rionegro', 'itagui', ..., 'bucarest', 'naranjal', 'goretty', 'palenque', 'jardin']
Length: 287
Categories (286, object): ['13', '20', 'abreo', 'abriaqui', ..., 'yumbo', 'zamora', 'zona', 'zuñiga']


### **Variables Númericas**

In [9]:
datos_enteros = datos_cooperativa.select_dtypes(include=['int64'])
print(datos_enteros.columns)

Index(['plazo', 'vinculacion', 'saldo_interes', 'aportes', 'garantias',
       'valorgarantia', 'reestr', 'tipoasociado', 'estado_cliente', 'sexo',
       'actualizacion', 'default', 'grupo_dptmto', 'grupo_ciudad',
       'grupo_edad', 'grupo_actividadeco'],
      dtype='object')


In [10]:
datos_flotantes = datos_cooperativa.select_dtypes(include=['float64'])
print(datos_flotantes.columns)

Index(['valor_cuota', 'valor_prestamo', 'saldo_capital', 'ctasahorros', 'edad',
       'curtotalingresos', 'curtotalegresos', 'intestrato', 'puntaje_data'],
      dtype='object')


En las variables númericas que se encuentran `valorgarantia` y `saldo_interes` las vamos a convertir a tipo de dato flotante ya que representan valores monetarios reales, por lo tanto pueden llegar a contener decimales.

In [11]:
datos_cooperativa["saldo_interes"] = datos_cooperativa["saldo_interes"].astype(float)
datos_cooperativa["valorgarantia"] = datos_cooperativa["valorgarantia"].astype(float)

In [12]:
datos_flotantes = datos_cooperativa.select_dtypes(include=['float64'])
print(datos_flotantes.columns)

Index(['valor_cuota', 'valor_prestamo', 'saldo_capital', 'saldo_interes',
       'valorgarantia', 'ctasahorros', 'edad', 'curtotalingresos',
       'curtotalegresos', 'intestrato', 'puntaje_data'],
      dtype='object')


Las variables `edad` e `intestrato` se convertirán a tipo entero, ya que representan valores discretos y no requieren decimales para su procesamiento.


In [13]:
datos_cooperativa["edad"] = datos_cooperativa["edad"].astype(int)
datos_cooperativa["intestrato"] = datos_cooperativa["intestrato"].astype(int)

In [14]:
datos_enteros = datos_cooperativa.select_dtypes(include=['int64'])
print(datos_enteros.columns)

Index(['plazo', 'vinculacion', 'aportes', 'garantias', 'reestr',
       'tipoasociado', 'estado_cliente', 'sexo', 'actualizacion', 'default',
       'grupo_dptmto', 'grupo_ciudad', 'grupo_edad', 'grupo_actividadeco'],
      dtype='object')


La variable `default` se convertirá a tipo booleano, ya que representa una variable binaria.

In [15]:
datos_cooperativa["default"] = datos_cooperativa["default"].astype(bool)
datos_bool = datos_cooperativa.select_dtypes(include=['bool'])
print(datos_bool.columns)

Index(['default'], dtype='object')


Verificamos que los tipos de datos se encuentren correctos después de ajustarlos

In [16]:
datos_cooperativa.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12910 entries, 0 to 12909
Data columns (total 28 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   cartera             12910 non-null  category
 1   plazo               12910 non-null  int64   
 2   vinculacion         12910 non-null  int64   
 3   valor_cuota         12910 non-null  float64 
 4   valor_prestamo      12910 non-null  float64 
 5   saldo_capital       12910 non-null  float64 
 6   saldo_interes       12910 non-null  float64 
 7   aportes             12910 non-null  int64   
 8   garantias           12910 non-null  int64   
 9   valorgarantia       12910 non-null  float64 
 10  reestr              12910 non-null  int64   
 11  ctasahorros         12910 non-null  float64 
 12  edad                12910 non-null  int32   
 13  tipoasociado        12910 non-null  int64   
 14  estado_cliente      12910 non-null  int64   
 15  departamento        12909 non-null  

## ➜ **Eliminación de Variables**

La variable `ciudad` se eliminó porque tiene demasiadas categorías diferentes (287 ciudades). Esto aumenta mucho la cantidad de columnas al transformar los datos y puede hacer más difícil el entrenamiento del modelo.

Además, muchas ciudades tienen pocos registros, por lo que no aportan información suficiente para encontrar patrones útiles.

En cambio, las variables `departamento`, `grupo_dptmto` y `grupo_ciudad` se conservaron porque tienen una menor cantidad de categorías, permitiendo representar la información geográfica de forma más general y eficiente. En especial, `grupo_ciudad` se mantuvo porque también está relacionada con la variable `ciudad`, pero agrupando las ciudades en categorías más manejables para el modelo.




In [17]:
datos_cooperativa = datos_cooperativa.drop(columns=["ciudad"])
print(datos_cooperativa.columns)

Index(['cartera', 'plazo', 'vinculacion', 'valor_cuota', 'valor_prestamo',
       'saldo_capital', 'saldo_interes', 'aportes', 'garantias',
       'valorgarantia', 'reestr', 'ctasahorros', 'edad', 'tipoasociado',
       'estado_cliente', 'departamento', 'sexo', 'curtotalingresos',
       'curtotalegresos', 'intestrato', 'actualizacion', 'default',
       'puntaje_data', 'grupo_dptmto', 'grupo_ciudad', 'grupo_edad',
       'grupo_actividadeco'],
      dtype='object')


In [18]:
datos_cooperativa.sample(8, random_state=1000)

,cartera,plazo,vinculacion,valor_cuota,valor_prestamo,saldo_capital,saldo_interes,aportes,garantias,valorgarantia,...,curtotalingresos,curtotalegresos,intestrato,actualizacion,default,puntaje_data,grupo_dptmto,grupo_ciudad,grupo_edad,grupo_actividadeco
9780,consumo_sin_libranza,1096,827,166485.0,5100000.0,1551461.0,3720.0,304582,1,304582.0,...,1000000.0,200000.0,2,0,True,737.0,3,4,1,4
6445,consumo_sin_libranza,731,868,139299.0,2680000.0,1986847.0,127680.0,449372,1,449372.0,...,1300000.0,390000.0,3,1,True,693.0,3,7,1,4
8716,consumo_con_libranza,2192,3481,883296.0,36000000.0,21353348.0,0.0,5523320,1,5523320.0,...,4750000.0,1000000.0,2,0,False,834.0,3,7,2,4
3535,consumo_sin_libranza,2557,2419,3122994.0,136900000.0,122644820.0,2073568.0,5239946,2,197271672.0,...,13300000.0,4000000.0,6,0,False,787.0,3,6,3,4
8902,consumo_sin_libranza,1826,1706,747096.0,35000000.0,30328543.0,0.0,2102369,1,2102369.0,...,6000000.0,500000.0,2,0,True,748.0,3,2,1,4
8229,consumo_sin_libranza,1461,1288,306272.0,10100000.0,1735917.0,1910.0,363050,1,2506689.0,...,4972000.0,800000.0,2,1,False,775.0,3,6,1,4
8934,consumo_sin_libranza,1096,1939,212608.0,5930000.0,3043287.0,645077.0,429711,1,429711.0,...,1027739.0,150000.0,3,0,True,772.0,3,7,1,4
4616,consumo_sin_libranza,2191,2165,593968.0,21950000.0,21332507.0,373325.0,1886050,1,1886050.0,...,2020776.0,500000.0,2,1,False,771.0,3,7,2,4


## ➜ **Identificación de Valores Faltantes (Nan, Null, None)**

In [19]:
datos_cooperativa.isnull().sum()

cartera               0
plazo                 0
vinculacion           0
valor_cuota           0
valor_prestamo        0
saldo_capital         0
saldo_interes         0
aportes               0
garantias             0
valorgarantia         0
reestr                0
ctasahorros           0
edad                  0
tipoasociado          0
estado_cliente        0
departamento          1
sexo                  0
curtotalingresos      0
curtotalegresos       0
intestrato            0
actualizacion         0
default               0
puntaje_data          0
grupo_dptmto          0
grupo_ciudad          0
grupo_edad            0
grupo_actividadeco    0
dtype: int64

Eliminamos los valores nulos encontrados en la variable `departamento`.

Se eliminaron estos registros porque la cantidad de datos faltantes era mínima (solo 1 registro) en comparación con el tamaño total del dataset.

Imputar este valor podría introducir información incorrecta o poco representativa, ya que al ser una variable categórica geográfica no existe una forma confiable de inferir el departamento real del cliente.

Dado que la pérdida de datos es muy pequeña, fue más adecuado eliminar ese registro para mantener la calidad y consistencia del conjunto de datos sin afectar significativamente el análisis ni el rendimiento del modelo.



In [20]:
datos_cooperativa = datos_cooperativa.dropna(subset=['departamento'])
print(datos_cooperativa.isnull().sum())

cartera               0
plazo                 0
vinculacion           0
valor_cuota           0
valor_prestamo        0
saldo_capital         0
saldo_interes         0
aportes               0
garantias             0
valorgarantia         0
reestr                0
ctasahorros           0
edad                  0
tipoasociado          0
estado_cliente        0
departamento          0
sexo                  0
curtotalingresos      0
curtotalegresos       0
intestrato            0
actualizacion         0
default               0
puntaje_data          0
grupo_dptmto          0
grupo_ciudad          0
grupo_edad            0
grupo_actividadeco    0
dtype: int64


Verificamos finalmente el estado de los datos después de eliminar los valores nulos

In [21]:
datos_cooperativa.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12909 entries, 0 to 12909
Data columns (total 27 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   cartera             12909 non-null  category
 1   plazo               12909 non-null  int64   
 2   vinculacion         12909 non-null  int64   
 3   valor_cuota         12909 non-null  float64 
 4   valor_prestamo      12909 non-null  float64 
 5   saldo_capital       12909 non-null  float64 
 6   saldo_interes       12909 non-null  float64 
 7   aportes             12909 non-null  int64   
 8   garantias           12909 non-null  int64   
 9   valorgarantia       12909 non-null  float64 
 10  reestr              12909 non-null  int64   
 11  ctasahorros         12909 non-null  float64 
 12  edad                12909 non-null  int32   
 13  tipoasociado        12909 non-null  int64   
 14  estado_cliente      12909 non-null  int64   
 15  departamento        12909 non-null  categ

#  ➜ **Guardado de Datos para el EDA**

Para este paso vamos a guardar el dataset ajustado con sus tipos de datos correctos para realizar el EDA(Analisís Exploratorio de Datos)

In [22]:
print(Path.cwd())

c:\Users\belac\OneDrive\Documentos\Universidad\Ciencia de Datos y ML\Mlops\credit-risk-classification\notebooks


In [23]:
BASE_DIR = Path.cwd().parent

ruta_datos_procesados = BASE_DIR / "data" / "processed"

ruta_datos_procesados.mkdir(parents=True, exist_ok=True)

ruta_archivo = ruta_datos_procesados / "02_datos_ajustados_cooperativa.parquet"

datos_cooperativa.to_parquet(ruta_archivo, index=False)